# step07_npq_comparison - chlorophyll profiles: NPQ removed vs NPQ kept (P45_06)

A side-by-side view of what each cast's calibrated chlorophyll profile looks like under the two calibrations:

- **A, NPQ removed** (the step07 recommended product): the daytime near-surface quenched points are excluded
  from the fit.
- **B, NPQ kept** (the step07_noremoval variant): those points are retained in the fit.

The notebook computes BOTH calibrations itself from your data (so it does not depend on the other two notebooks
having been run), then applies each to every cast's fluorometer downcast and overlays the two corrected
profiles, with the extracted-chl bottles marked. The point is to SEE how much the choice of including the
quenched surface points changes the profiles.

**Reading the plots.** Grey is the raw sensor fluorescence (factory chl units). Blue is corrected with the
NPQ-removed calibration (A), orange with the NPQ-kept calibration (B). Red squares are extracted Chl-a bottles;
open red squares are the NPQ-flagged (daytime surface) bottles. Where the blue and orange lines diverge is where
the two calibrations disagree; they diverge most at high chlorophyll (the gain difference) and near the surface
(where quenching lives).

**Honest framing.** The ECO puck measures chlorophyll FLUORESCENCE, a proxy for Chl-a, so both profiles are
fluorescence-derived Chl-a estimates (Roesler et al. 2017). A is the defensible product because surface daytime
fluorescence is light-suppressed and reads low; B is a sensitivity check. This notebook does not pick one, it
shows the difference.

Sources: Roesler et al. (2017), *L&O: Methods*; Welschmeyer (1994), *L&O*; Sea-Bird WET Labs ECO documentation.

In [ ]:
from pathlib import Path
import re, numpy as np, pandas as pd, matplotlib
INTERACTIVE_ZOOM = True
try: get_ipython().run_line_magic("matplotlib","widget" if INTERACTIVE_ZOOM else "inline")
except Exception:
    INTERACTIVE_ZOOM=False
    try: get_ipython().run_line_magic("matplotlib","inline")
    except Exception: matplotlib.use("Agg")
import matplotlib.pyplot as plt
if INTERACTIVE_ZOOM:
    try: plt.ioff()
    except Exception: pass
try: from IPython.display import display, Image
except Exception:
    def display(o): print(o)
    Image=None

CRUISE_ID  = "P45_06"
CTD_ROOT   = Path(r"C:\Projects\ctd_pipeline")
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID
CTD_DIR    = CRUISE_DIR / "L2" / "CTD" / "CTD_1m_down"
CTD_SUFFIX = "_1m_down.cnv"
if not CTD_DIR.exists():
    CTD_DIR = CRUISE_DIR / "L1" / "CTD"; CTD_SUFFIX = ".cnv"
META       = CRUISE_DIR / "metadata"
PAIR_AUDIT = META / "bottle_nitrate" / f"{CRUISE_ID}_bottle_pairing_audit.csv"
CHLA_FILE  = META / "chla" / f"{CRUISE_ID}_chla.csv"
CHLA_GLOB  = ["*chla*.csv","*chl*.csv"]
OUT_DIR    = CRUISE_DIR / "_audit" / "productivity_compare"; OUT_DIR.mkdir(parents=True, exist_ok=True)

SENSOR_COLS = ["flECO-AFL","flECO-AFL1","flC"]
DEPTH_WIN   = 2.0
MIN_IN_WIN  = 3
NPQ_DEPTH   = 10.0
DAY_START, DAY_END = 7, 17
print("cruise:", CRUISE_ID, "| CTD:", CTD_DIR, "| out:", OUT_DIR)

In [ ]:
# ------------------------------------------------------------------ readers + two-calibration engine
def parse_cnv_downcast(path):
    lines=Path(path).read_text(encoding="latin-1",errors="replace").splitlines(); names=[]; iend=None
    for i,ln in enumerate(lines):
        if ln.startswith("# name"): names.append(ln.split("=",1)[1].split(":")[0].strip())
        if ln.strip().upper().startswith("*END*"): iend=i+1; break
    rows=[]
    for ln in lines[iend:]:
        p=ln.split()
        if len(p)>=len(names):
            try: rows.append([float(x) for x in p[:len(names)]])
            except ValueError: pass
    d=np.array(rows,float); d[(d<=-8e-29)&(d>=-1.2e-28)]=np.nan
    df=pd.DataFrame(d,columns=names)
    pcol=next((c for c in names if re.search(r"pr[dD]M|prM|pressure|depS",c)),None)
    imax=int(np.nanargmax(df[pcol].to_numpy())) if pcol else len(df)-1
    return names, df.iloc[:imax+1].reset_index(drop=True), pcol

def load_station_cast():
    if not PAIR_AUDIT.exists(): return {}
    a=pd.read_csv(PAIR_AUDIT); cols={c.lower():c for c in a.columns}
    s=cols.get("station"); c=cols.get("matched_cast") or cols.get("cast_id")
    return {str(r[s]).strip():str(r[c]).strip() for _,r in a.iterrows()} if (s and c) else {}

def find_ctd_files():
    return {p.name[:-len(CTD_SUFFIX)]:p for p in sorted(CTD_DIR.glob(f"*{CTD_SUFFIX}"))}

def _resolve_chla():
    if CHLA_FILE.exists(): return CHLA_FILE
    folder=CHLA_FILE.parent
    if not folder.exists(): return None
    for pat in CHLA_GLOB:
        hits=sorted(folder.glob(pat))+sorted(folder.glob(pat.upper()))
        if hits: return hits[0]
    return None

def load_chla(s2c):
    f=_resolve_chla()
    if f is None: return None
    d=pd.read_csv(f); d.columns=[c.strip() for c in d.columns]; low={c.lower():c for c in d.columns}
    vcol=next((low[v] for v in ["chl","chl_a","chla","chlorophyll","chlorophyll_a"] if v in low),None)
    dcol=next((low[v] for v in ["depth","depth_m","deps","pressure"] if v in low),None)
    out=pd.DataFrame(dict(value=pd.to_numeric(d[vcol],errors="coerce"),depth_m=pd.to_numeric(d[dcol],errors="coerce")))
    out["station"]=d[low["station"]].astype(str).str.strip() if "station" in low else ""
    ccol=low.get("cast_id") or low.get("cast")
    cid=d[ccol].astype(str).str.strip() if ccol else pd.Series([""]*len(d))
    out["cast_id"]=cid.where(cid.str.startswith("P45"), out["station"].map(s2c))
    tc=low.get("date_time") or low.get("time_utc")
    out["time"]=pd.to_datetime(d[tc],errors="coerce") if tc else pd.NaT
    return out.dropna(subset=["value","depth_m","cast_id"])

def sensor_at(df,pcol,scol,z):
    m=df[(df[pcol]>=z-DEPTH_WIN)&(df[pcol]<=z+DEPTH_WIN)]
    v=m[scol].to_numpy(); v=v[np.isfinite(v)]
    return (float(np.median(v)),len(v)) if len(v) else (np.nan,0)

def calibrate(exclude_npq):
    """Return dict(offset,gain,R2,n,cal_df) for chl, excluding or keeping NPQ points."""
    disc=load_chla(S2C)
    disc=disc.groupby(["cast_id","station","depth_m"],as_index=False).agg(
        value=("value","mean"), time=("time","first"))
    rows=[]
    for _,r in disc.iterrows():
        path=CTDF.get(r["cast_id"])
        if path is None:
            rows.append(dict(cast=r["cast_id"],depth_m=r["depth_m"],bottle=r["value"],sensor=np.nan,npq=False,used=False)); continue
        names,ddf,pcol=parse_cnv_downcast(path)
        scol=next((c for c in SENSOR_COLS if c in names),None)
        sv,n=sensor_at(ddf,pcol,scol,float(r["depth_m"])) if scol else (np.nan,0)
        hour=r["time"].hour if pd.notna(r["time"]) else 12
        npq=bool(r["depth_m"]<=NPQ_DEPTH and DAY_START<=hour<DAY_END)
        used=(n>=MIN_IN_WIN) and not (npq and exclude_npq)
        rows.append(dict(cast=r["cast_id"],depth_m=r["depth_m"],bottle=r["value"],sensor=sv,npq=npq,used=used))
    cal=pd.DataFrame(rows); fit=cal[cal["used"]].dropna(subset=["sensor"])
    off=gn=r2=rmse_raw=rmse_corr=factory=np.nan
    if len(fit)>=2 and np.ptp(fit["bottle"])>0:
        b=fit["bottle"].to_numpy(); s=fit["sensor"].to_numpy()
        gn,off=np.polyfit(b,s,1); pred=off+gn*b
        r2=1-np.sum((s-pred)**2)/np.sum((s-s.mean())**2)
        corr_at=(s-off)/gn
        rmse_raw=float(np.sqrt(np.mean((s-b)**2))); rmse_corr=float(np.sqrt(np.mean((corr_at-b)**2)))
        factory=1.0/gn if gn else np.nan
    return dict(offset=off,gain=gn,R2=r2,n=len(fit),rmse_raw=rmse_raw,rmse_corr=rmse_corr,factory=factory,cal=cal,fit=fit)

S2C=load_station_cast(); CTDF=find_ctd_files()
A=calibrate(exclude_npq=True)     # NPQ removed  (recommended = step07)
B=calibrate(exclude_npq=False)    # NPQ kept     (sensitivity  = step07_noremoval)
print(f"{len(CTDF)} CTD casts\n")
# ---- side-by-side comparison table (A vs B) ----
def _row(name,C): return dict(variant=name, n=C["n"], offset=round(C["offset"],3), gain=round(C["gain"],3),
                              R2=round(C["R2"],3), factory_to_true=round(C["factory"],3),
                              rmse_raw=round(C["rmse_raw"],3), rmse_corr=round(C["rmse_corr"],3))
CMP=pd.DataFrame([_row("A  NPQ removed (step07)",A), _row("B  NPQ kept (noremoval)",B)])
with pd.option_context("display.width",200): print(CMP.to_string(index=False))
if np.isfinite(A["gain"]) and np.isfinite(B["gain"]):
    dg=B["gain"]-A["gain"]
    print(f"\n  chl_corr = (fluor - offset) / gain")
    print(f"  A -> B:  gain {dg:+.3f} ({100*dg/A['gain']:+.0f}%),  offset {B['offset']-A['offset']:+.3f} mg/m3,  "
          f"R2 {B['R2']-A['R2']:+.3f},  corrected-RMSE {B['rmse_corr']-A['rmse_corr']:+.3f} mg/m3")
CMP.to_csv(OUT_DIR/f"{CRUISE_ID}__npq_comparison_table.csv",index=False,encoding="utf-8-sig")
print("  saved", f"{CRUISE_ID}__npq_comparison_table.csv")

In [ ]:
# ------------------------------------------------------------------ comparison figures
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg

def _apply(ddf,pcol,scol,C): return (ddf[scol]-C["offset"])/C["gain"]

def cast_png_bytes(cid):
    import io
    path=CTDF.get(cid)
    fig=Figure(figsize=(5.2,6.4)); FigureCanvasAgg(fig); ax=fig.subplots()
    if path is not None and np.isfinite(A["offset"]) and np.isfinite(B["offset"]):
        names,ddf,pcol=parse_cnv_downcast(path); scol=next((c for c in SENSOR_COLS if c in names),None)
        if scol:
            ax.plot(ddf[scol],ddf[pcol],".",ms=2,color="#c9c9c9",label="raw fluorescence")
            ax.plot(_apply(ddf,pcol,scol,A),ddf[pcol],"-",lw=1.3,color="#2E6FB0",label="A: NPQ removed")
            ax.plot(_apply(ddf,pcol,scol,B),ddf[pcol],"-",lw=1.3,color="#E07B39",label="B: NPQ kept")
        bd=A["cal"][A["cal"]["cast"]==cid]
        good=bd[~bd["npq"]]; q=bd[bd["npq"]]
        if len(good): ax.plot(good["bottle"],good["depth_m"],"s",ms=8,color="#d62728",label="bottle Chl-a",zorder=5)
        if len(q): ax.plot(q["bottle"],q["depth_m"],"s",ms=9,mfc="none",mec="#d62728",label="bottle (NPQ-flagged)",zorder=5)
        hi=max(2.0,float(np.nanmax(bd["bottle"]))*1.6) if len(bd) else 5.0
        ax.set_xlim(-0.3,hi)
    ax.invert_yaxis(); ax.axvline(0,color="#bbb",lw=0.6)
    ax.set_xlabel("Chl-a (fluorescence-derived, mg/m3)"); ax.set_ylabel("pressure (db)")
    ax.set_title(f"{cid}   A gain {A['gain']:.2f} vs B gain {B['gain']:.2f}",fontsize=10)
    ax.legend(fontsize=7,frameon=False)
    for s in ("top","right"): ax.spines[s].set_visible(False)
    fig.tight_layout()
    buf=io.BytesIO(); fig.savefig(buf,format="png",dpi=120); buf.seek(0)
    try: plt.close(fig)
    except Exception: pass
    return buf.getvalue()

# (1) all-cast grid saved to disk
casts=[c for c in CTDF if (A["cal"]["cast"]==c).any()] or list(CTDF)
ncol=4; nrow=int(np.ceil(len(casts)/ncol))
fig=Figure(figsize=(3.0*ncol,4.2*nrow)); FigureCanvasAgg(fig); axes=fig.subplots(nrow,ncol,squeeze=False).ravel()
for ax,cid in zip(axes,casts):
    path=CTDF.get(cid)
    if path is None: ax.set_visible(False); continue
    names,ddf,pcol=parse_cnv_downcast(path); scol=next((c for c in SENSOR_COLS if c in names),None)
    if scol and np.isfinite(A["offset"]):
        ax.plot(_apply(ddf,pcol,scol,A),ddf[pcol],"-",lw=1.0,color="#2E6FB0")
        ax.plot(_apply(ddf,pcol,scol,B),ddf[pcol],"-",lw=1.0,color="#E07B39")
    bd=A["cal"][A["cal"]["cast"]==cid]
    if len(bd): ax.plot(bd["bottle"],bd["depth_m"],"s",ms=5,color="#d62728")
    ax.invert_yaxis(); ax.set_title(cid.replace("P45_06_",""),fontsize=8)
    ax.set_xlim(-0.3,max(2.0,float(np.nanmax(bd["bottle"]))*1.6) if len(bd) else 5.0)
    for s in ("top","right"): ax.spines[s].set_visible(False)
for ax in axes[len(casts):]: ax.set_visible(False)
fig.suptitle(f"{CRUISE_ID}  Chl-a profiles: blue = NPQ removed (A), orange = NPQ kept (B), red = bottles",fontsize=11)
fig.tight_layout(rect=[0,0,1,0.97])
gridpng=OUT_DIR/f"{CRUISE_ID}__chl_npq_comparison_allcasts.png"; fig.savefig(gridpng,dpi=130)
try: plt.close(fig)
except Exception: pass
if Image is not None:
    try: display(Image(filename=str(gridpng)))
    except Exception: pass
print("  saved",gridpng.name)

In [ ]:
# ------------------------------------------------------------------ interactive: per-cast comparison
try:
    import ipywidgets as W; _HAVE_W=True
except Exception:
    _HAVE_W=False; print("ipywidgets not installed; pip install ipywidgets then Restart + Run All.")
if _HAVE_W and CTDF:
    _casts=[c for c in CTDF if (A["cal"]["cast"]==c).any()] or list(CTDF)
    wc=W.Dropdown(options=_casts,value=_casts[0],description="cast")
    def _r(cid): display(Image(data=cast_png_bytes(cid)))
    out=W.interactive_output(_r,{"cid":wc}); display(W.VBox([wc,out]))